In [22]:
import time, os, argparse, random, copy, sys
if not sys.warnoptions:
    import warnings
    warnings.simplefilter("ignore")

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay
# import seaborn as sns
# sns.set()
from sklearn import tree
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
import seaborn as sns
from imblearn.over_sampling import SMOTE, ADASYN
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

import tensorflow as tf
from torch.utils.tensorboard import SummaryWriter
import torch, math
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print("Importing libraries")

Importing libraries


In [2]:
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"Using GPU: {physical_devices[0]}")
else:
    raise SystemError("GPU device not found. Please ensure your GPU is properly installed and available.")

Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


I0000 00:00:1737670389.031814   18469 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1737670389.040043   18469 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1737670389.042541   18469 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355


# Test tensorflow training on gpu

In [4]:
# Load CIFAR-10 dataset (example dataset)
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

# Define the CNN model
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10)
])

# Compile the model
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Train the model
model.fit(train_images, train_labels, epochs=15,
          validation_data=(test_images, test_labels), batch_size=64)

# Evaluate the model
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"Test accuracy: {test_acc}")

2025-01-02 11:48:42.360082: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 614400000 exceeds 10% of free system memory.
2025-01-02 11:48:42.645929: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 614400000 exceeds 10% of free system memory.


Epoch 1/15


I0000 00:00:1735847323.746339  304380 service.cc:146] XLA service 0x7b5e5c0041f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1735847323.746358  304380 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 2060, Compute Capability 7.5
2025-01-02 11:48:43.768064: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-01-02 11:48:43.864731: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90101


101/782 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1951 - loss: 2.1599

I0000 00:00:1735847325.048760  304380 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3458 - loss: 1.7762

2025-01-02 11:48:48.001889: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 122880000 exceeds 10% of free system memory.
2025-01-02 11:48:48.056578: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 122880000 exceeds 10% of free system memory.


782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.3459 - loss: 1.7759 - val_accuracy: 0.5431 - val_loss: 1.2855
Epoch 2/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5555 - loss: 1.2540 - val_accuracy: 0.5834 - val_loss: 1.1636
Epoch 3/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6076 - loss: 1.1046 - val_accuracy: 0.6266 - val_loss: 1.0548
Epoch 4/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6512 - loss: 0.9848 - val_accuracy: 0.6127 - val_loss: 1.0904
Epoch 5/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6809 - loss: 0.9107 - val_accuracy: 0.6682 - val_loss: 0.9354
Epoch 6/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7077 - loss: 0.8319 - val_accuracy: 0.6784 - val_loss: 0.9187
Epoch 7/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7195 - loss: 0.8005 - val_accuracy: 0.6833 - val_loss: 0.9139
Epoch 8/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7413 - loss: 0.7402 - val_accuracy: 0.6882 - val_

2025-01-02 11:49:19.694554: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 122880000 exceeds 10% of free system memory.


313/313 - 1s - 2ms/step - accuracy: 0.7111 - loss: 0.8992
Test accuracy: 0.7110999822616577


# Laad data

In [23]:
filenames = {
    "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
    # "wisconsin_ssd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_ssd_unmerged_V3.csv",

    "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
    # "wisconsin_hdd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_hdd_unmerged_V3.csv",

    "wisconsin_hdd_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv",
    # "wisconsin_hdd_ssd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-hdd-ssd_unmerged_V3.csv",

    "wisconsin_ssd_delay_10ms_merged":"./ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv",
    # "wisconsin_ssd_delayed_10ms_unmerged":"./ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_unmerged_V3.csv"

    "wisconsin_hdd_delay_10ms_merged":"./ds/v3/selected_cols_merged/wisconsin-220g2-hdd-delayed-10ms_merged_V3.csv",
    # # "wisconsin_hdd_delay_10ms_unmerged":"./ds/v3/selected_cols/wisconsin-220g2-hdd-delayed-10ms_unmerged_V3.csv",

    "utah_ssd_merged": "./ds/v3/selected_cols_merged/utah-6525-25g-25Gbps_ssd_merged.csv",
    # "utah_ssd_unmerged": "./ds/v3/selected_cols/utah-6525-25g-25Gbps_ssd_unmerged_V3.csv",

    "utah_ssd_delay_30ms_merged":"./ds/v3/selected_cols_merged/utah-6525-25-ssd-delayed-30ms_merged_V3.csv",
    # "utah_ssd_delay_unmerged": "./ds/v3/selected_cols/utah-6525-25-ssd-delayed-30ms_unmerged_V3.csv",


    "utah_ssd_delay_10ms_merged":"./ds/v3/selected_cols_merged/utah-6525-25-ssd-delayed-10ms_merged_V3.csv",
    # # "utah_ssd_delay_10ms_merged": "./ds/v3/selected_cols/utah-6525-25-ssd-delayed-10ms_unmerged_V3.csv",

}
data = {}

In [24]:
def remove_labels_in_df(df, labels_values):
    for lbl in labels_values:
        df = df.drop(df[df.label_value == lbl].index)
    return df

In [25]:
def normalize_df(df):
    df['sender_avg_rtt_value'] = df['sender_avg_rtt_value'] / df[df.label_value == 0].sender_avg_rtt_value.mean()
    df['sender_retrans'] = df['sender_retrans'] / df[df.label_value == 0].sender_seg_out.mean()
    # df["sender_avg_send_value"] = df["sender_avg_send_value"] / df[df.label_value == 0].sender_avg_send_value.mean()
    df["sender_segs_in"] = df["sender_segs_in"] / df[df.label_value == 0].sender_segs_in.mean()

    # df["sender_ost_read"] = df["sender_ost_read"] / df[df.label_value == 0].sender_ost_read.mean()

    df["sender_nic_send_bytes"] = df["sender_nic_send_bytes"] / df[df.label_value == 0].sender_nic_send_bytes.mean()
    df["sender_nic_receive_bytes"] = df["sender_nic_receive_bytes"] / df[df.label_value == 0].sender_nic_receive_bytes.mean()

    df["sender_remote_ost_read_bytes"] =df["sender_remote_ost_read_bytes"] / df[df.label_value == 0].sender_remote_ost_read_bytes.mean()

    # df["receiver_segs_in"] = df["receiver_segs_in"] / df[df.label_value == 0].receiver_segs_in.mean()
    df["receiver_seg_out"] = df["receiver_seg_out"] / df[df.label_value == 0].receiver_seg_out.mean()

    # df["receiver_write_bytes"] = df["receiver_write_bytes"] / df[df.label_value == 0].receiver_write_bytes.mean()
    # df["receiver_ost_write"] = df["receiver_ost_write"] / df[df.label_value == 0].receiver_ost_write.mean()

    df["receiver_nic_send_bytes"] = df["receiver_nic_send_bytes"] / df[df.label_value == 0].receiver_nic_send_bytes.mean()
    df["receiver_nic_receive_bytes"] = df["receiver_nic_receive_bytes"] / df[df.label_value == 0].receiver_nic_receive_bytes.mean()

    df["receiver_remote_ost_write_bytes"] = df["receiver_remote_ost_write_bytes"] / df[df.label_value == 0].receiver_remote_ost_write_bytes.mean()

    df["sender_tcp_snd_buffer_max"] = df["sender_tcp_snd_buffer_max"] / df[df.label_value == 0].sender_tcp_snd_buffer_max.mean()
    df["receiver_tcp_rcv_buffer_max"] = df["receiver_tcp_rcv_buffer_max"] / df[df.label_value == 0].receiver_tcp_rcv_buffer_max.mean()

    # df["sender_write_bytes_io"] = df["sender_write_bytes_io"] / df[df.label_value == 0].sender_write_bytes_io.mean()
    # df["sender_read_bytes_io"] = df["sender_read_bytes_io"] / df[df.label_value == 0].sender_read_bytes_io.mean()
    #
    # df["receiver_read_bytes_io"] = df["receiver_read_bytes_io"] / df[df.label_value == 0].receiver_read_bytes_io.mean()
    # df["receiver_write_bytes_io"] = df["receiver_write_bytes_io"] / df[df.label_value == 0].receiver_write_bytes_io.mean()

    #---------------
    # df["sender_ssthresh_value"] = df.sender_ssthresh_value / df.sender_cwnd_rate
    # df["sender_req_active"] = df["sender_req_active"] / df[df.label_value == 0].sender_req_active.mean()
    return df

In [85]:
wisconsin_ssd_merged = pd.read_csv(filenames.get('wisconsin_ssd_merged')).drop(['time_stamp', 'through_put'], axis=1)

In [78]:

print(wisconsin_ssd_merged.shape)
df = wisconsin_ssd_merged
print(len(dict(Counter(df[df.columns[len(df.columns) - 1]]))))

(8402, 145)
13


In [84]:
features =  ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max', 'sender_nic_send_bytes', 'sender_nic_receive_bytes', 'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']
print(len(features))

12


In [86]:
df = wisconsin_ssd_merged
remove_labels = [17, 21, 25, 29]
# df = remove_labels_in_df(df, remove_labels)
X = df.drop(columns="label_value")[features] # df[features]
y = df.label_value
X_train, X_test, y_train, y_test = train_test_split(X,y)
X_train, y_train = RandomOverSampler(sampling_strategy="all").fit_resample(X_train, y_train)
clf = MLPClassifier()
clf = make_pipeline(StandardScaler(), clf)
# X_train = StandardScaler().fit_transform(X_train)
# X_test = StandardScaler().fit_transform(X_test)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
accuracy = np.round(metrics.f1_score(y_test, y_pred, average='weighted') * 100, 2)
print(accuracy)

98.86


In [88]:
print((len(dict(Counter(df[df.columns[len(df.columns) - 1]])))))
print(Counter(y))

13
Counter({37: 1035, 43: 1033, 13: 694, 17: 694, 29: 693, 25: 692, 21: 692, 33: 689, 10: 521, 1: 520, 7: 517, 4: 449, 0: 173})


In [89]:
class MLPClassifier_torch(nn.Module):
    def __init__(self, input_size, output_size=2, hidden_layer_sizes=(100,),
                 learning_rate=0.001, max_iter=200, tol=1e-4, random_state=None):
        super(MLPClassifier_torch, self).__init__()

        if random_state is not None:
            torch.manual_seed(random_state)

        # Create the network architecture
        layers = []
        prev_size = input_size
        for size in hidden_layer_sizes:
            layers.append(nn.Linear(prev_size, size))
            # layers.append(nn.BatchNorm1d(size))
            layers.append(nn.ReLU())
            prev_size = size
        layers.append(nn.Linear(prev_size, output_size))
        # layers.append(nn.Softmax(dim=1))  # Softmax for multi-class classification

        self.model = nn.Sequential(*layers)
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.tol = tol
        self.optimizer = None
        # self.criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss for multi-class log loss
        self.criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss for multi-class log loss

    def forward(self, x):
        return self.model(x)

    def fit(self, X, y, batch_size='auto', verbose=False):
        # Ensure input is a numpy array
        if isinstance(X, np.ndarray) is False:
            X = X.to_numpy()
        if isinstance(y, np.ndarray) is False:
            y = y.to_numpy()
        # Prepare data
        dataset = TensorDataset(torch.tensor(X, dtype=torch.float32),
                                 torch.tensor(y, dtype=torch.long))  # Integer labels for CrossEntropyLoss
        if batch_size == 'auto':
            batch_size = min(200, len(dataset))  # Following sklearn's default
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        # Set up optimizer
        self.optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)

        # Training loop
        for epoch in range(self.max_iter):
            epoch_loss = 0.0
            for batch_X, batch_y in loader:
                # self.optimizer.zero_grad()
                self.zero_grad()
                outputs = self.forward(batch_X)
                loss = self.criterion(outputs, batch_y)
                loss.backward()
                self.optimizer.step()
                epoch_loss += loss.item()

            # Compute average loss for the epoch
            epoch_loss /= len(loader)

            if verbose:
                print(f"Epoch {epoch + 1}/{self.max_iter}, Loss: {epoch_loss}")

            # Check for convergence
            if epoch_loss < self.tol:
                if verbose:
                    print("Convergence reached.")
                break

    def predict(self, X):
        # Ensure input is a numpy array
        if isinstance(X, np.ndarray) is False:
            X = X.to_numpy()
        self.eval()  # Set to evaluation mode
        with torch.no_grad():
            outputs = self.forward(torch.tensor(X, dtype=torch.float32))
            predictions = torch.argmax(outputs, dim=1)  # Get the class with the highest probability
        return predictions.numpy()


In [93]:
df = wisconsin_ssd_merged
remove_labels = [17, 21, 25, 29]
encoder = LabelEncoder()
scaler = StandardScaler()
# df = remove_labels_in_df(df, remove_labels)
X = df.drop(columns="label_value")[features] # df[features]
y = df.label_value
y = encoder.fit_transform(y)
label_mapping = {index: label for index, label in enumerate(encoder.classes_)}

X_train, X_test, y_train, y_test = train_test_split(X,y)

# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

X_train, y_train = RandomOverSampler(sampling_strategy="all").fit_resample(X_train, y_train)

clf = MLPClassifier_torch(input_size=X.shape[1], output_size=len(np.unique(y)),
                          learning_rate=0.001)
print(clf)
clf.fit(X_train, y_train, verbose=True)

MLPClassifier_torch(
  (model): Sequential(
    (0): Linear(in_features=12, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=13, bias=True)
  )
  (criterion): CrossEntropyLoss()
)
Epoch 1/200, Loss: 18066635.429245282
Epoch 2/200, Loss: 1493950.0742924528
Epoch 3/200, Loss: 1206923.8372641508
Epoch 4/200, Loss: 1023871.5188679246
Epoch 5/200, Loss: 1118292.4882075472
Epoch 6/200, Loss: 885791.6851415094
Epoch 7/200, Loss: 878437.4528301887
Epoch 8/200, Loss: 860279.3944575472
Epoch 9/200, Loss: 967737.2676886793
Epoch 10/200, Loss: 759638.4233490566
Epoch 11/200, Loss: 1034814.6214622641
Epoch 12/200, Loss: 573860.3968160377
Epoch 13/200, Loss: 632318.3295990566
Epoch 14/200, Loss: 501344.68396226416
Epoch 15/200, Loss: 794061.929245283
Epoch 16/200, Loss: 788588.78125
Epoch 17/200, Loss: 615666.3425707547
Epoch 18/200, Loss: 487381.36202830187
Epoch 19/200, Loss: 641641.1751179246
Epoch 20/200, Loss: 448311.5728183962
Epoch 21/200, Loss: 63843

In [95]:
y_pred = clf.predict(X_test)
# print(y_pred)
# accuracy = np.round(metrics.f1_score(y_test, y_pred, average='weighted') * 100, 2)
accuracy = np.round(metrics.accuracy_score(y_test, y_pred) * 100, 2)
print(accuracy)

67.54


# Try centralized traiing using all datasets together

In [127]:
def get_centralize_dataloaders(args, remove_labels=None, features=None, filenames=None):
    if remove_labels is None:
        remove_labels = [17, 21, 25, 29]
    if features is None:
        features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
            'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']
    if filenames is None:
        filenames = {
            "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
            # "wisconsin_ssd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_ssd_unmerged_V3.csv",

            "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
            # "wisconsin_hdd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_hdd_unmerged_V3.csv",
        }
    
    # We will accumulate train and test data from all files here
    X_train_all, y_train_all = [], []
    X_test_all, y_test_all = [], []

    encoder = LabelEncoder()
    scaler = StandardScaler()


    for client_name, file_path in filenames.items():
        # Step 1: Load the dataset and Label encoding and scaling
        df = pd.read_csv(file_path)
        df = remove_labels_in_df(df, remove_labels)
        # Normalize the data for transfer learning
        # df = normalize_df(df)

        X = df.drop(columns="label_value")[features]
        y = df.label_value

        encoder = LabelEncoder()
        # scaler = StandardScaler()

        y = encoder.fit_transform(y)

        # Step 3: Split into train and test sets
        X_train, X_test, y_train, y_test = train_test_split(X,y)

        # Step 4: Over-sample the minority class
        X_train, y_train = RandomOverSampler(sampling_strategy="all").fit_resample(X_train, y_train)


        X_train = X_train.to_numpy() if not isinstance(X_train, np.ndarray) else X_train
        X_test = X_test.to_numpy() if not isinstance(X_test, np.ndarray) else X_test
        y_train = y_train.to_numpy() if not isinstance(y_train, np.ndarray) else y_train
        y_test = y_test.to_numpy() if not isinstance(y_test, np.ndarray) else y_test

        # 7) Append to our global lists
        X_train_all.append(X_train)
        y_train_all.append(y_train)
        X_test_all.append(X_test)
        y_test_all.append(y_test)

    # 8) Concatenate all the data
    X_train_all = np.concatenate(X_train_all, axis=0)
    y_train_all = np.concatenate(y_train_all, axis=0)
    X_test_all = np.concatenate(X_test_all, axis=0)
    y_test_all = np.concatenate(y_test_all, axis=0)

    # 9) Create TensorDatasets
    train_dataset = TensorDataset(
        torch.tensor(X_train_all, dtype=torch.float32),
        torch.tensor(y_train_all, dtype=torch.long)
    )
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)

    test_dataset = TensorDataset(
        torch.tensor(X_test_all, dtype=torch.float32),
        torch.tensor(y_test_all, dtype=torch.long)
    )
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=True)

    total_classes = len(torch.unique(test_loader.dataset[:][1]))
    return train_loader, test_loader, total_classes


def central_train(net, ldr_train, epochs: int, device, verbose=False, local_lr=0.001):

    # optimizer = optim.Adam(net.parameters(), lr=local_lr, betas=(0.9, 0.999), eps=1e-08)
    epochs_losses = []
    net.train()
    loss_func = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=local_lr)

    # TODO implement dynamic lr
    for epoch in range(epochs):
        correct, total, epoch_loss = 0, 0, 0.0
        for _, (batch_X, labels) in enumerate(ldr_train):
            batch_X, labels = batch_X.to(device), labels.to(device)
            net.zero_grad()
            # optimizer.zero_grad()
            log_probs = net.forward(batch_X)
            loss = loss_func(log_probs, labels)
            loss.backward()
            optimizer.step()
            # Metrics
            epochs_losses.append(loss.item())
            epoch_loss += loss.item()
            total += labels.size(0)
            correct += (torch.max(log_probs.data, 1)[1] == labels).sum().item()
        if verbose and epoch % 10 == 0:
            epoch_acc = correct / total
            epoch_loss /= len(ldr_train.dataset)
            print(f"\tEpoch {epoch}: train loss {epoch_loss}, accuracy {epoch_acc}")
    w_new = copy.deepcopy(net.state_dict())
    return w_new, net, sum(epochs_losses) / len(epochs_losses)


def central_test(net, ldr_test, device):
    
    net = copy.deepcopy(net).to(device)
    loss_func = nn.CrossEntropyLoss()
    net.eval()
    correct, total, test_loss = 0, 0, 0.0
    
    all_preds, all_targets = [], []

    with torch.no_grad():
        for index, (data, target) in enumerate(ldr_test):
             data, target = data.to(device), target.to(device)
             log_probs = net.forward(data)
             test_loss += loss_func(log_probs, target).item()
             _, predicted = torch.max(log_probs, -1) # TODO CHECK FOR GET -1 pr 1 is correct
             
             total += target.size(0)
             correct += predicted.eq(target).sum()
             all_preds.extend(predicted.cpu().numpy())
             all_targets.extend(target.cpu().numpy())
    test_loss /= len(ldr_test.dataset)
    accuracy = 100.00 * correct.item() / total
    
    f1 = f1_score(all_targets, all_preds, average='weighted')
    return test_loss, accuracy, f1

In [135]:
parser = argparse.ArgumentParser()
parser.add_argument('--gpu',
                    type=int,
                    default=0,
                    help="GPU ID, -1 for CPU")
parser.add_argument('--seed',
                    type=int,
                    default=1,
                    help="seed")
parser.add_argument('--repeat', type=int, default=1, help='repeat index')
meta_args = parser.parse_args("")
meta_args.device = torch.device('cuda:{}'.format(meta_args.gpu) if torch.cuda.is_available() and meta_args.gpu != -1 else 'cpu')
meta_args.log_path = "centerilized"
meta_args.model = "mlp"

meta_args.round = 1 # 50
meta_args.epoch_iterations = 20
meta_args.batch_size = 150
meta_args.decay_weight = 1.0
meta_args.data_type = ""

meta_args.local_lr = 0.003 # with simple adam no betas 81% at round 31 and gets ~87% but train lossalso much lower than above case
meta_args.min_local_lr = 1e-08
meta_args.decay_weight = 0.85


meta_args.remove_labels = [17, 21, 25, 29]
# meta_args.remove_labels = []
meta_args.features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
            'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']  
meta_args.filenames = { 
    "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
    "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
    "wisconsin_hdd_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv",
    "wisconsin_ssd_delay_10ms_merged":"./ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv",
    
    "wisconsin_hdd_delay_10ms_merged":"./ds/v3/selected_cols_merged/wisconsin-220g2-hdd-delayed-10ms_merged_V3.csv",
    "utah_ssd_merged": "./ds/v3/selected_cols_merged/utah-6525-25g-25Gbps_ssd_merged.csv",
    "utah_ssd_delay_30ms_merged":"./ds/v3/selected_cols_merged/utah-6525-25-ssd-delayed-30ms_merged_V3.csv",
    "utah_ssd_delay_10ms_merged":"./ds/v3/selected_cols_merged/utah-6525-25-ssd-delayed-10ms_merged_V3.csv",
    }

train_ldr, test_ldr, total_classes = get_centralize_dataloaders(args=args, remove_labels=args.remove_labels, features=args.features, filenames=args.filenames)

meta_args.input_size = len(meta_args.features)
meta_args.output_size = total_classes

args = copy.deepcopy(meta_args)

num_runs = 5
accuracies = []
f1_scores = []
losses = []

for run in range(num_runs):
    print(f"Run {run + 1}/{num_runs}")
    
    # Reset the model and other variables if necessary
    clf = MLPClassifier_torch(input_size=args.input_size, output_size=args.output_size, hidden_layer_sizes=(250,)).to(args.device)
    
    lr = args.local_lr
    for r in range(args.round):
        lr = lr * args.decay_weight if args.decay_weight < 1.0 and lr > args.min_local_lr else args.min_local_lr
        print(f"Round: {r}, learning rate {lr}")
        _, _, loss = central_train(clf, train_ldr, epochs=args.epoch_iterations, device=args.device, verbose=False, local_lr=lr)
        loss, accuracy, f1 = central_test(clf, test_ldr, device=args.device)
        print(f"loss {loss}, accuracy {accuracy}, f1_score {f1}")
    
        accuracies.append(accuracy)
        f1_scores.append(f1)
        losses.append(loss)

# Calculate average values
avg_accuracy = np.mean(accuracies)
avg_f1_score = np.mean(f1_scores)
avg_loss = np.mean(losses)

# Print per run values and average values
for run in range(num_runs):
    print(f"Run {run + 1}: loss {losses[run]}, accuracy {accuracies[run]}, f1_score {f1_scores[run]}")

print(f"\nAverage: loss {avg_loss}, accuracy {avg_accuracy}, f1_score {avg_f1_score}")

Run 1/5
Round: 0, learning rate 0.00255
loss 24.97919397546243, accuracy 78.6925710645752, f1_score 0.7926544050446249
Run 2/5
Round: 0, learning rate 0.00255
loss 26.140371091052202, accuracy 75.65889003901584, f1_score 0.7672887932099381
Run 3/5
Round: 0, learning rate 0.00255
loss 70.38143297619611, accuracy 86.69480054144438, f1_score 0.8735369431355815
Run 4/5
Round: 0, learning rate 0.00255
loss 77.11147295896183, accuracy 85.4845130981766, f1_score 0.8657567794419524
Run 5/5
Round: 0, learning rate 0.00255
loss 21.07661916088342, accuracy 70.41961939644877, f1_score 0.7258475661270996
Run 1: loss 24.97919397546243, accuracy 78.6925710645752, f1_score 0.7926544050446249
Run 2: loss 26.140371091052202, accuracy 75.65889003901584, f1_score 0.7672887932099381
Run 3: loss 70.38143297619611, accuracy 86.69480054144438, f1_score 0.8735369431355815
Run 4: loss 77.11147295896183, accuracy 85.4845130981766, f1_score 0.8657567794419524
Run 5: loss 21.07661916088342, accuracy 70.41961939644

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [125]:
def get_data_loader_list(args, remove_labels=None, features=None, filenames=None):
    if remove_labels is None:
        remove_labels = [17, 21, 25, 29]
    if features is None:
        features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
                    'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']
    if filenames is None:
        filenames = {
            "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
            # "wisconsin_ssd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_ssd_unmerged_V3.csv",

            "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
            # "wisconsin_hdd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_hdd_unmerged_V3.csv",
        }
    clients_data_loaders = {}
    test_data_dict = {}


    
    for client_name, file_path in filenames.items():
        # Step 1: Load the dataset and Label encoding and scaling
        df = pd.read_csv(file_path)
        df = remove_labels_in_df(df, remove_labels)
        # Normalize for transfer learning 
        # df = normalize_df(df)

        X = df.drop(columns="label_value")[features]
        y = df.label_value

        encoder = LabelEncoder()
        scaler = StandardScaler()
        
        y = encoder.fit_transform(y)

        # Step 3: Split into train and test sets
        # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        X_train, X_test, y_train, y_test = train_test_split(X,y)
        
       
        # X_train = scaler.fit_transform(X_train)
        # X_test = scaler.transform(X_test)

        # Step 4: Apply oversampling to training data
        X_train, y_train = RandomOverSampler(sampling_strategy="all").fit_resample(X_train, y_train)

        X_train = X_train.to_numpy() if not isinstance(X_train, np.ndarray) else X_train
        X_test = X_test.to_numpy() if not isinstance(X_test, np.ndarray) else X_test
        y_train = y_train.to_numpy() if not isinstance(y_train, np.ndarray) else y_train
        y_test = y_test.to_numpy() if not isinstance(y_test, np.ndarray) else y_test


        # Step 5: Create train DataLoader
        train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                    torch.tensor(y_train, dtype=torch.long))

        ldr_train = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
        # data_loader_list.append(ldr_train)
        clients_data_loaders[client_name] = ldr_train
        test_data_dict[client_name] = (X_test, y_test)

    return clients_data_loaders, test_data_dict

In [112]:
def create_test_loader(args, test_data_dict):
    client_test_loaders = {}
    combined_X_test, combined_y_test = [], []
    for client_name in test_data_dict.keys():
        X_test, y_test = test_data_dict[client_name]
        X_test = X_test.to_numpy() if not isinstance(X_test, np.ndarray) else X_test
        y_test = y_test.to_numpy() if not isinstance(y_test, np.ndarray) else y_test

        # Combine test data for unified test dataset
        combined_X_test.append(X_test)
        combined_y_test.append(y_test)

        # Create individual test DataLoader
        test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                      torch.tensor(y_test, dtype=torch.long))
        
        client_test_loaders[client_name] = DataLoader(test_dataset, batch_size=args.batch_size)
    # Combine all test data
    combined_X_test = np.vstack(combined_X_test)
    combined_y_test = np.hstack(combined_y_test)
    total_classes = len(np.unique(combined_y_test))
     # Create combined test DataLoader
    combined_test_dataset = TensorDataset(torch.tensor(combined_X_test, dtype=torch.float32),
                                           torch.tensor(combined_y_test, dtype=torch.long))
    combined_test_loader = DataLoader(combined_test_dataset, batch_size=args.batch_size, shuffle=False)
    return combined_test_loader, client_test_loaders, total_classes        


In [113]:
def model_dim(model):
    flat = [torch.flatten(model[k]) for k in model.keys()]
    s = 0
    for p in flat: 
        s += p.shape[0]
    return s

def model_setup(args):
    if args.model == 'mlp':
        # net_glob = MLPClassifier_torch(input_size=args.input_size, output_size=args.output_size).to(args.device)
        net_glob = MLPClassifier_torch(input_size=args.input_size, output_size=args.output_size, hidden_layer_sizes=(200,)).to(args.device)
        # net_glob = MLPClassifier_torch(input_size=args.input_size, output_size=args.output_size, hidden_layer_sizes=(100,100,)).to(args.device)
    else:
        raise ValueError("Model not supported")
    global_model = copy.deepcopy(net_glob.state_dict())
    return args, net_glob, global_model, model_dim(global_model)

In [114]:
def set_log_path(args):
    import datetime
    path =  './log/' + args.log_path+ '/'
    if not os.path.exists(path):
        os.makedirs(path)
    path_log = os.path.join(path)
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    return path_log + '_' + str(timestamp)

In [115]:
def average(global_model, local_updates):
    '''
    simple average
    '''
    model_update = {k: local_updates[0][k] *0.0 for k in local_updates[0].keys()}
    for i in range(len(local_updates)):
        model_update = {k: model_update[k] +  local_updates[i][k] for k in global_model.keys()}
    global_model = {k: global_model[k] +  model_update[k]/ len(local_updates) for k in global_model.keys()}
    return global_model

In [116]:
def test(net_g, ldr_test, args):
    net_g = copy.deepcopy(net_g).to(args.device)
    loss_func = nn.CrossEntropyLoss()
    net_g.eval()
    test_loss = 0
    correct = 0
    all_preds = []
    all_targets = []
    
    for index, (data, target) in enumerate(ldr_test):
        data, target = data.to(args.device), target.to(args.device)
        log_probs = net_g.forward(data)
        test_loss += loss_func(log_probs, target).item()
        _, predicted = torch.max(log_probs, -1)
        correct += predicted.eq(target).sum()
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(target.cpu().numpy())
    
    test_loss /= len(ldr_test.dataset)
    accuracy = 100.00 * correct.item() / len(ldr_test.dataset)
    f1 = f1_score(all_targets, all_preds, average='weighted')

    return accuracy, test_loss, f1

In [117]:
class LocalUpdate(object):
    def __init__(self, args):
        self.args = args
        if args.data_type == 'image':
            self.loss_func = nn.CrossEntropyLoss()
        elif args.data_type == 'text':
            self.loss_func = nn.CrossEntropyLoss()
        else:
            self.loss_func = nn.CrossEntropyLoss()
    
    def local_sgd(self, net, ldr_train):
        # optimizer = torch.optim.SGD(net.parameters(), lr=self.args.local_lr)
        optimizer = torch.optim.SGD(net.parameters(), lr=self.args.local_lr, momentum=0.5)
        epoch_loss = []
        net.train()
        for epoch in range(self.args.epoch_iterations):
            for _, (batch_X, labels) in enumerate(ldr_train):
                batch_X, labels = batch_X.to(self.args.device), labels.to(self.args.device)
                net.zero_grad()
                # optimizer.zero_grad()
                log_probs = net.forward(batch_X)
                loss = self.loss_func(log_probs, labels)
                loss.backward()
                optimizer.step()
                epoch_loss.append(loss.item())
        w_new = copy.deepcopy(net.state_dict())
        return w_new, sum(epoch_loss) / len(epoch_loss)

    def local_sgd_adam(self, net, ldr_train):
        optimizer = optim.Adam(net.parameters(), lr=self.args.local_lr)
        # optimizer = optim.Adam(net.parameters(), lr=self.args.local_lr, betas=(0.5, 0.5))
        epoch_loss = []
        net.train()
        for epoch in range(self.args.epoch_iterations):
            for _, (batch_X, labels) in enumerate(ldr_train):
                batch_X, labels = batch_X.to(self.args.device), labels.to(self.args.device)
                net.zero_grad()
                # optimizer.zero_grad()
                # log_probs = net(batch_X)
                log_probs = net.forward(batch_X)
                loss = self.loss_func(log_probs, labels)
                loss.backward()
                optimizer.step()
                epoch_loss.append(loss.item())
        w_new = copy.deepcopy(net.state_dict())
        return w_new, sum(epoch_loss) / len(epoch_loss)

In [122]:
def fedavg(args):
    print("{:<50}".format("-" * 15 + " data setup " + "-" * 50)[0:60])
    print("total clients: ", len(filenames))

    print("{:<50}".format("-" * 15 + " log path " + "-" * 50)[0:60])
    log_path = set_log_path(args)
    writer = SummaryWriter(log_path)
    print(log_path)
    
    print("{:<50}".format("-" * 15 + " data loader " + "-" * 50)[0:60])
    clients_data_loaders, test_data_dict = get_data_loader_list(args, remove_labels=args.remove_labels, features=args.features, filenames=args.filenames)
    test_loader, client_test_loaders, total_classes = create_test_loader(args, test_data_dict)

    args.input_size = len(features)
    args.output_size = total_classes
    

    print("{:<50}".format("-" * 15 + " model setup " + "-" * 50)[0:60])
    args, net_glob, global_model, args.dim = model_setup(args)
    net_glob = net_glob.to(args.device)

    # local_model = local_model.to(device)
    print('model dim:', args.dim)

    print(net_glob.parameters())
    print("{:<50}".format("-" * 15 + " training " + "-" * 50)[0:60])
    
    net_glob.train()

    for t in range(args.round):
        t1 = time.time()
        print(f"round {t}")
        ## learning rate decaying
        args.local_lr = args.local_lr * args.decay_weight


        net_glob.load_state_dict(global_model)
         ## local training
        local_solver = LocalUpdate(args=args)
        local_losses, local_updates, delta_norms= [], [], []
        for client in clients_data_loaders.keys():
            local_model, local_loss = local_solver.local_sgd_adam(
                net=copy.deepcopy(net_glob).to(args.device),
                ldr_train=clients_data_loaders[client])
            # local_model, local_loss = local_solver.local_sgd(
            #     net=copy.deepcopy(net_glob).to(args.device),
            #     ldr_train=clients_data_loaders[client])
            local_losses.append(local_loss)
            # compute model update
            model_update = {k: local_model[k] - global_model[k] for k in global_model.keys()}
            # model_update = {k: local_model[k].to(args.device) - global_model[k].to(args.device) for k in global_model.keys()}

            # compute model update norm
            delta_norm = torch.norm(torch.cat([torch.flatten(model_update[k])for k in model_update.keys()]))
            delta_norms.append(delta_norm)
            # clipping local model 
            # threshold = delta_norm / args.clip
            # if threshold > 1.0:
                # for k in model_update.keys():
                    # model_update[k] = model_update[k] / threshold
            # collecting local models
            local_updates.append(model_update)
        
        # metrics
        norm = torch.median(torch.stack(delta_norms)).cpu()
        train_loss = sum(local_losses) / len(local_losses)
        writer.add_scalar('norm', norm, t)
        writer.add_scalar('train_loss', train_loss, t)
        
        # global aggregation
        global_model = average(global_model, local_updates)
        # ## test global model on server side
        net_glob.load_state_dict(global_model)
        net_glob.eval()
        test_acc, test_loss, f1 = test(net_glob, test_loader, args)
        # metrics
        writer.add_scalar('test_acc', test_acc, t)
        writer.add_scalar('F1 score', f1, t)
        # test_acc.append(test_acc_t)
        print('t {:3d}: train_loss = {:.3f}, norm = {:.3f}, test_acc = {:.3f}, f1_score = {:.3f}'.
              format(t, train_loss, norm, test_acc, f1))

        #  stop
        if math.isnan(train_loss) or train_loss > 1e8 or t == args.round - 1:
            # np.savetxt(log_path + "_test_acc" + ".csv",
            #             test_acc,
            #             delimiter=",")
            # np.savetxt(log_path + "_train_loss" + ".csv",
            #             train_loss,
            #             delimiter=",")
            # np.savetxt(log_path + "_norm_" + ".csv", median_model_norm, delimiter=",")
            t2 = time.time()
            hours, rem = divmod(t2-t1, 3600)
            minutes, seconds = divmod(rem, 60)
            print("training time: {:0>2}:{:0>2}:{:05.2f}".format(int(hours),int(minutes),seconds))
            # exit()

    return None

In [123]:
parser = argparse.ArgumentParser()
parser.add_argument('--gpu',
                    type=int,
                    default=0,
                    help="GPU ID, -1 for CPU")
parser.add_argument('--seed',
                    type=int,
                    default=1,
                    help="seed")
parser.add_argument('--repeat', type=int, default=1, help='repeat index')
meta_args = parser.parse_args("")
meta_args.device = torch.device('cuda:{}'.format(meta_args.gpu) if torch.cuda.is_available() and meta_args.gpu != -1 else 'cpu')
meta_args.log_path = "fed_avg"
meta_args.model = "mlp"

# meta_args.model = "cnn"SO FAR GOOD WITHOUT NORMALIZATION 
# meta_args.round = 20
# meta_args.epoch_iterations = 20
# meta_args.local_lr = 0.001
# meta_args.batch_size = 100
# meta_args.decay_weight = 1.0
# meta_args.data_type = ""
meta_args.round = 80 # 50
meta_args.epoch_iterations = 20
meta_args.local_lr = 0.001
meta_args.batch_size = 150
meta_args.decay_weight = 1.0
meta_args.data_type = ""

meta_args.remove_labels = [17, 21, 25, 29]
meta_args.features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
            'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']  
meta_args.filenames = { 
    "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
    "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
    "wisconsin_hdd_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv",
    "wisconsin_ssd_delay_10ms_merged":"./ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv",
    }

args = copy.deepcopy(meta_args)
clients_data_loaders, test_data_dict = get_data_loader_list(args, remove_labels=args.remove_labels, features=args.features, filenames=args.filenames)

test_data_dict['wisconsin_ssd_merged']


print("Train data distribution")
# Preing the label distribution over each dataset in the train_data_dict, Sorted by the label value
for client_name in clients_data_loaders.keys():
    print(client_name, dict(sorted(Counter(clients_data_loaders[client_name].dataset.tensors[1].numpy()).items())))

print("Test data distribution")
# Preing the label distribution over each dataset in the test_data_dict Sorted by the label
for client_name in test_data_dict.keys():
    print(client_name, dict(sorted(Counter(test_data_dict[client_name][1]).items())))




Train data distribution
wisconsin_ssd_merged {0: 786, 1: 786, 2: 786, 3: 786, 4: 786, 5: 786, 6: 786, 7: 786, 8: 786}
wisconsin_hdd_merged {0: 998, 1: 998, 2: 998, 3: 998, 4: 998, 5: 998, 6: 998, 7: 998, 8: 998}
wisconsin_hdd_ssd_merged {0: 991, 1: 991, 2: 991, 3: 991, 4: 991, 5: 991, 6: 991, 7: 991, 8: 991}
wisconsin_ssd_delay_10ms_merged {0: 787, 1: 787, 2: 787, 3: 787, 4: 787, 5: 787, 6: 787, 7: 787, 8: 787}
Test data distribution
wisconsin_ssd_merged {0: 49, 1: 122, 2: 107, 3: 147, 4: 128, 5: 174, 6: 166, 7: 268, 8: 247}
wisconsin_hdd_merged {0: 60, 1: 164, 2: 155, 3: 164, 4: 171, 5: 211, 6: 227, 7: 340, 8: 315}
wisconsin_hdd_ssd_merged {0: 62, 1: 166, 2: 143, 3: 158, 4: 155, 5: 238, 6: 205, 7: 343, 8: 325}
wisconsin_ssd_delay_10ms_merged {0: 46, 1: 140, 2: 118, 3: 131, 4: 128, 5: 164, 6: 160, 7: 259, 8: 269}


In [124]:
parser = argparse.ArgumentParser()
parser.add_argument('--gpu',
                    type=int,
                    default=0,
                    help="GPU ID, -1 for CPU")
parser.add_argument('--seed',
                    type=int,
                    default=1,
                    help="seed")
parser.add_argument('--repeat', type=int, default=1, help='repeat index')
meta_args = parser.parse_args("")
meta_args.device = torch.device('cuda:{}'.format(meta_args.gpu) if torch.cuda.is_available() and meta_args.gpu != -1 else 'cpu')
meta_args.log_path = "fed_avg"
meta_args.model = "mlp"

# meta_args.model = "cnn"SO FAR GOOD WITHOUT NORMALIZATION 
# meta_args.round = 20
# meta_args.epoch_iterations = 20
# meta_args.local_lr = 0.001
# meta_args.batch_size = 100
# meta_args.decay_weight = 1.0
# meta_args.data_type = ""
meta_args.round = 1 # 50
meta_args.epoch_iterations = 20
meta_args.local_lr = 0.001
meta_args.batch_size = 150
meta_args.decay_weight = 1.0
meta_args.data_type = ""

meta_args.remove_labels = [17, 21, 25, 29]
meta_args.features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
            'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']  
meta_args.filenames = { 
    "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
    "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
    "wisconsin_hdd_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv",
    "wisconsin_ssd_delay_10ms_merged":"./ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv",
    }

print(meta_args)
for r in range(meta_args.repeat):
    r = r + 1
    args = copy.deepcopy(meta_args)
    random.seed(args.seed+r)
    torch.manual_seed(args.seed+r)
    np.random.seed(args.seed+r)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    best_result = fedavg(args)


Namespace(gpu=0, seed=1, repeat=1, device=device(type='cuda', index=0), log_path='fed_avg', model='mlp', round=1, epoch_iterations=20, local_lr=0.001, batch_size=150, decay_weight=1.0, data_type='', remove_labels=[17, 21, 25, 29], features=['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max', 'sender_nic_send_bytes', 'sender_nic_receive_bytes', 'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes'], filenames={'wisconsin_ssd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv', 'wisconsin_hdd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv', 'wisconsin_hdd_ssd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv', 'wisconsin_ssd_delay_10ms_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv'})
--------------- data se

In [1]:
import os
import sys
os.path.dirname(sys.executable)

'/home/ehsan/miniconda3/envs/flower/bin'